# GE NBG — National Bank of Georgia

Jira: DECD-4822. The NBG site is a Next.js app. Entity data comes from three source types — **no Selenium needed**:

1. **`__NEXT_DATA__`** JSON embedded in the page (single-list pages: commercial banks, microbanks).
2. **JSON API** `https://nbg.gov.ge/gw/api/pg/pages/static/<key>/organizations` with header `Accept-Language: en` (tabbed supervision pages).
3. **Excel registers** linked on the non-bank-institutions page (microfinance, credit unions, loan-issuing, currency-exchange).

| ListCode | ListName | Source | Note |
|---|---|---|---|
| 1 | Licensed Commercial Banks | __NEXT_DATA__ | |
| 2 | Licensed Microbanks | __NEXT_DATA__ | |
| 3 | Microfinance organizations | Excel (EN) | |
| 4 | Credit Unions | Excel (EN) | |
| 5 | Loan Issuing Entities | Excel (KA) | head offices only; translated |
| 6 | Currency Exchange Units | Excel (KA) | head offices only; translated |
| 7 | Brokerage Companies | API | |
| 8 | Securities Registrars | API | |
| 9 | Stock Exchanges | API | |
| 10 | Central Depository | API | |
| 11 | Investment Funds | API | |
| 12 | Asset Management Companies | API | |
| 13 | Specialized Depositaries | API | may be empty |

L5/L6 are Georgian: `Name` keeps the original Georgian, English translation goes to `Name - Mother Company`; `City`/`Address_2`(region)/`CoType`(legal form) are translated. Only **head offices** (სათაო) are kept, not branches (ფილიალი).

In [1]:
# ------------------------------------------------ Import Lib ----------------------------------------
import os
import re
import json
import datetime
from time import sleep
from urllib.parse import quote, urlsplit, urlunsplit

import requests
import pandas as pd
from bs4 import BeautifulSoup

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ------------------------------------------------ Begin_fileName ----------------------------------------
regulatorName = 'GE NBG'
print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(':', '.')[:-7])
processdate = now.strftime('%Y-%m-%d')

try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__))
except NameError:
    scriptfolder = os.path.join(r"C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator", regulatorName)
os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder')
if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

HEADERS = {'User-Agent': 'Mozilla/5.0', 'Accept-Language': 'en'}
API_TMPL = 'https://nbg.gov.ge/gw/api/pg/pages/static/{}/organizations'
NONBANK_URL = 'https://nbg.gov.ge/en/page/non-bank-institutions'

Running GE NBG Web Scraping Tool v.1.0


In [2]:
# ------------------------------------------------ Begin_Variable ----------------------------------------
sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
           'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
           'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
           'RegCtry': [], 'RegCode': [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
           'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
           'Phone - Mother company': []}

# Per-list configuration. type: nextdata | api | content_table | excel_en | excel_ka
LISTS = [
    {'code': '1',  'name': 'Licensed Commercial Banks', 'type': 'api', 'key': 'licensedCommercialBanks', 'fieldset': 'bank', 'language': 'English'},
    {'code': '2',  'name': 'Licensed Microbanks',        'type': 'api', 'key': 'licensedMicrobanks',        'fieldset': 'bank', 'language': 'English'},
    {'code': '3',  'name': 'List of microfinance organizations registered in Georgia', 'type': 'excel_en', 'label': 'Microfinance organizations', 'sheet': 'MFI',
     'cols': {'name': 2, 'regno': 1, 'regno_type': 'Registration No.', 'legalform': 3, 'region': 6, 'city': 7, 'address': 8, 'website': 9, 'idcode': 10, 'idcode_type': 'Identification Code'}, 'language': 'English'},
    {'code': '4',  'name': 'List of Credit Unions licensed in Georgia', 'type': 'excel_en', 'label': 'Credit Unions', 'sheet': 'Credit Unions',
     'cols': {'name': 2, 'regno': 1, 'regno_type': 'Licence No.', 'legalform': 3, 'region': 6, 'city': 7, 'address': 8, 'idcode': 9, 'idcode_type': 'Identification Number'}, 'language': 'English'},
    {'code': '5',  'name': 'List of Loan Issuing Entities', 'type': 'excel_ka', 'label': 'Loan Issuing Entities', 'sheet': 'სგს რეესტრი',
     'cols': {'status': 2, 'name': 3, 'regno': 1, 'regno_type': 'Registration No.', 'legalform': 4, 'actdate': 6, 'region': 7, 'city': 8, 'address': 9, 'idcode': 10, 'idcode_type': 'Identification Code'}, 'language': 'Georgian'},
    {'code': '6',  'name': 'List of Currency Exchange Units', 'type': 'excel_ka', 'label': 'Currency Exchange Units', 'sheet': 'რეგისტრირებული',
     'cols': {'status': 2, 'name': 3, 'regno': 1, 'regno_type': 'Registration No.', 'legalform': 4, 'actdate': 6, 'region': 7, 'city': 8, 'address': 9, 'idcode': 10, 'idcode_type': 'Identification Code'}, 'language': 'Georgian'},
    {'code': '7',  'name': 'Brokerage Companies',       'type': 'api', 'key': 'licensedParticipantsBrokerageCompanies',       'language': 'English'},
    {'code': '8',  'name': 'Securities Registrars',     'type': 'api', 'key': 'licensedParticipantsSecuritiesRegistrars',     'language': 'English'},
    {'code': '9',  'name': 'Stock Exchanges',           'type': 'api', 'key': 'licensedParticipantsLicensedStockExchange',    'language': 'English'},
    {'code': '10', 'name': 'Central Depository',        'type': 'api', 'key': 'licensedParticipantsLicensedCentralDepository','language': 'English'},
    {'code': '11', 'name': 'Investment Funds',          'type': 'api', 'key': 'investmentFundsInvestmentFunds',                'language': 'English'},
    {'code': '12', 'name': 'Asset Management Companies','type': 'api', 'key': 'investmentFundsInvestmentFundManagementCompany','language': 'English'},
    # L13 data is an HTML table inside the content/details description (the /organizations endpoint is empty for this key).
    {'code': '13', 'name': 'Specialized Depositaries',  'type': 'content_table', 'key': 'investmentFundsSpecializedDepositories', 'language': 'English'},
]

# ------------------------------------------------ Begin_Function ----------------------------------------
def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key in sqldict:
        if len(sqldict[key]) != maxlen:
            sqldict[key] = sqldict[key] + [''] * (maxlen - len(sqldict[key]))
    return sqldict


def add_entity(fields):
    """Append one entity. Only provided keys are set; the rest are padded blank."""
    fields.setdefault('ListProcessDate', processdate)
    fields.setdefault('RegCtry', 'GE')
    fields.setdefault('RegCode', 'NBG')
    fields.setdefault('Cntry', 'Georgia')
    fields.setdefault('RegulationType', 'Regulated')
    for k, v in fields.items():
        if k in sqldict:
            sqldict[k].append('' if v is None else str(v).strip())
    bourange_same_length_array(sqldict)


# ---- translation via Google gtx endpoint (verify=False handles the corporate self-signed cert) ----
_tcache = {}

def translate_ka(text, src='ka', dest='en', max_retries=4):
    if text is None:
        return ''
    s = re.sub(r'\s+', ' ', str(text)).strip()
    if s == '' or s.lower() in ('nan', 'none'):
        return ''
    if s in _tcache:
        return _tcache[s]
    url = 'https://translate.googleapis.com/translate_a/single'
    params = {'client': 'gtx', 'sl': src, 'tl': dest, 'dt': 't', 'q': s}
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(url, params=params, headers={'User-Agent': 'Mozilla/5.0'}, timeout=20, verify=False)
            data = r.json()
            out = ''.join(seg[0] for seg in data[0] if seg and seg[0]).strip()
            if out:
                _tcache[s] = out
                return out
        except Exception as e:
            if attempt == max_retries:
                print(f"[translate] giving up on {s!r}: {e}")
            sleep(1.5 * attempt)
    _tcache[s] = s  # fallback: keep original
    return s


def parse_contact(ci):
    """Split a free-text contactInfo string into (phone, email, website)."""
    phone = email = web = ''
    if not ci:
        return phone, email, web
    phones = []
    for p in re.split(r'[;\n]', str(ci)):
        p = p.strip()
        if not p:
            continue
        if '@' in p and ' ' not in p:
            email = p
        elif re.search(r'https?://|www\.', p):
            web = p
        else:
            phones.append(p)
    return '; '.join(phones), email, web


def get_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=40, verify=False)
    r.encoding = 'utf-8'
    return r.text


def get_nextdata_orgs(url):
    html = get_soup(url)
    m = re.search(r'<script id="__NEXT_DATA__"[^>]*>(.*?)</script>', html, re.S)
    data = json.loads(m.group(1))
    return data['props']['initialProps']['pageProps']['organizations']


def get_api_orgs(key):
    r = requests.get(API_TMPL.format(key), headers=HEADERS, timeout=40, verify=False)
    if r.status_code != 200:
        return []
    d = r.json()
    return d if isinstance(d, list) else []


def get_content_table_rows(key):
    """Parse the HTML <table> embedded in a tab's content/details `description` field.

    Used by lists whose entities are published as a content table rather than via
    the /organizations endpoint (e.g. Specialized Depositaries). Returns a list of
    cell-text rows, header row excluded.
    """
    u = 'https://nbg.gov.ge/gw/api/pg/pages/static/{}/content/details'.format(key)
    d = requests.get(u, headers=HEADERS, timeout=40, verify=False).json()
    soup = BeautifulSoup(d.get('description') or '', 'html.parser')
    out = []
    for i, tr in enumerate(soup.find_all('tr')):
        cells = [c.get_text(' ', strip=True) for c in tr.find_all(['td', 'th'])]
        if i == 0 or not cells or not cells[0]:
            continue
        out.append(cells)
    return out


def download_excel(label):
    """Resolve the current xlsx/xls link on the non-bank page by its English anchor text."""
    soup = BeautifulSoup(get_soup(NONBANK_URL), 'html.parser')
    href = None
    for a in soup.find_all('a', href=True):
        if '.xls' in a['href'].lower() and label.lower() in a.get_text(' ', strip=True).lower():
            href = a['href']
            break
    if not href:
        raise RuntimeError('Excel link not found for label: ' + label)
    p = urlsplit(href)
    enc = urlunsplit((p.scheme, p.netloc, quote(p.path), p.query, p.fragment))
    ext = '.xlsx' if href.lower().endswith('xlsx') else '.xls'
    path = os.path.join(tempfolder, 'list_' + re.sub(r'\W+', '_', label) + ext)
    rr = requests.get(enc, headers=HEADERS, timeout=90, verify=False)
    with open(path, 'wb') as f:
        f.write(rr.content)
    return path


def is_data_row(v):
    return pd.to_numeric(pd.Series([v]), errors='coerce').notna().iloc[0]

In [3]:
# ------------------------------------------------ Begin_Main ----------------------------------------
for cfg in LISTS:
    before = len(sqldict['Name'])
    common = {'ListCode': cfg['code'], 'ListName': cfg['name'], 'ListLanguage': cfg['language']}

    if cfg['type'] in ('nextdata', 'api'):
        orgs = get_nextdata_orgs(cfg['url']) if cfg['type'] == 'nextdata' else get_api_orgs(cfg['key'])
        for o in orgs:
            name = (o.get('title') or '').strip()
            if not name:
                continue
            if re.search(r'[Ⴀ-ჿ]', name):  # source served Georgian -> translate to English
                name = translate_ka(name)
            row = dict(common, Name=name)
            rd = o.get('releaseDate')
            if rd:
                row['RegulationDate'] = str(rd)[:10]
            if cfg.get('fieldset') == 'bank':
                if o.get('licenseNumberBank'):
                    row['InternalID_1'] = o['licenseNumberBank']
                    row['InternalID_1_type'] = 'License No.'
                if o.get('link'):
                    row['Website'] = o['link']
            else:  # api
                if o.get('identificationCode'):
                    row['InternalID_1'] = o['identificationCode']
                    row['InternalID_1_type'] = 'Identification Code'
                if o.get('licenseOrOrderNumber'):
                    row['InternalID_2'] = o['licenseOrOrderNumber']
                    row['InternalID_2_type'] = 'License / Order No.'
                phone, email, web = parse_contact(o.get('contactInfo'))
                if phone:
                    row['Phone'] = phone
                if email:
                    row['Email'] = email
                if web or o.get('link'):
                    row['Website'] = web or o.get('link')
            add_entity(row)

    elif cfg['type'] == 'content_table':
        # Entities published as an HTML table: Name | ID | Order Number | Order Date | Granted Status | Contact
        for cells in get_content_table_rows(cfg['key']):
            name = re.sub(r'\s*\*+\s*$', '', cells[0]).strip()  # drop trailing footnote asterisk
            if not name:
                continue
            row = dict(common, Name=name)
            if len(cells) > 1 and cells[1]:
                row['InternalID_1'] = cells[1]
                row['InternalID_1_type'] = 'Identification Code'
            if len(cells) > 2 and cells[2]:
                row['InternalID_2'] = cells[2]
                row['InternalID_2_type'] = 'Order Number'
            if len(cells) > 3 and cells[3]:
                row['RegulationDate'] = cells[3]
            if len(cells) > 5:
                phone, email, web = parse_contact(cells[5])
                if phone:
                    row['Phone'] = phone
                if email:
                    row['Email'] = email
                if web:
                    row['Website'] = web
            add_entity(row)

    elif cfg['type'] in ('excel_en', 'excel_ka'):
        path = download_excel(cfg['label'])
        df = pd.ExcelFile(path).parse(cfg['sheet'], header=None)
        c = cfg['cols']
        for _, r in df.iterrows():
            if not is_data_row(r[0]):
                continue
            if cfg['type'] == 'excel_ka' and str(r[c['status']]).strip() != 'სათაო':
                continue  # head offices only
            name = str(r[c['name']]).strip()
            if not name or name.lower() == 'nan':
                continue
            row = dict(common)
            row['Address_1'] = '' if pd.isna(r[c['address']]) else str(r[c['address']]).strip()
            if c.get('regno') is not None and not pd.isna(r[c['regno']]):
                row['InternalID_1'] = str(r[c['regno']]).strip()
                row['InternalID_1_type'] = c['regno_type']
            if c.get('idcode') is not None and not pd.isna(r[c['idcode']]):
                row['InternalID_2'] = str(r[c['idcode']]).strip()
                row['InternalID_2_type'] = c['idcode_type']
            if c.get('actdate') is not None and not pd.isna(r[c['actdate']]):
                row['RegulationDate'] = str(r[c['actdate']])[:10]
            if cfg['type'] == 'excel_en':
                row['Name'] = name
                row['City'] = '' if pd.isna(r[c['city']]) else str(r[c['city']]).strip()
                row['Address_2'] = '' if pd.isna(r[c['region']]) else str(r[c['region']]).strip()
                row['CoType'] = '' if pd.isna(r[c['legalform']]) else str(r[c['legalform']]).strip()
                if c.get('website') is not None and not pd.isna(r[c['website']]):
                    row['Website'] = str(r[c['website']]).strip()
            else:  # excel_ka: keep Georgian Name, translate into target columns
                row['Name'] = name
                row['Name - Mother Company'] = translate_ka(name)
                row['City'] = translate_ka(r[c['city']])
                row['Address_2'] = translate_ka(r[c['region']])
                row['CoType'] = translate_ka(r[c['legalform']])
            add_entity(row)

    after = len(sqldict['Name'])
    print(f"[INFO] List {cfg['code']:>2} {cfg['name'][:42]:<42} collected {after - before} rows")

[INFO] List  1 Licensed Commercial Banks                  collected 17 rows


[INFO] List  2 Licensed Microbanks                        collected 2 rows


[INFO] List  3 List of microfinance organizations registe collected 28 rows


[INFO] List  4 List of Credit Unions licensed in Georgia  collected 1 rows


[INFO] List  5 List of Loan Issuing Entities              collected 137 rows


...

[INFO] List  9 Stock Exchanges                            collected 2 rows


[INFO] List 10 Central Depository                         collected 1 rows


[INFO] List 11 Investment Funds                           collected 24 rows


[INFO] List 12 Asset Management Companies                 collected 11 rows


[INFO] List 13 Specialized Depositaries                   collected 3 rows


In [4]:
# ------------------------------------------------ Begin_writer and save df to excel ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
df = df[df['Name'].astype(str).str.strip() != '']
df.to_excel(filename, 'SQL Ready', index=False)
print('Saved {} rows -> {}'.format(len(df), filename))
df.groupby(['ListCode', 'ListName'])['Name'].count()

C:\Users\wuj1\AppData\Local\Temp\1\claude\ipykernel_44180\2881161987.py:5: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(filename, 'SQL Ready', index=False)


Saved 740 rows -> GE NBG SQL Ready 2026-06-05 14.23.06.xlsx


ListCode  ListName                                                
1         Licensed Commercial Banks                                    17
10        Central Depository                                            1
11        Investment Funds                                             24
12        Asset Management Companies                                   11
13        Specialized Depositaries                                      3
2         Licensed Microbanks                                           2
3         List of microfinance organizations registered in Georgia     28
4         List of Credit Unions licensed in Georgia                     1
5         List of Loan Issuing Entities                               137
6         List of Currency Exchange Units                             500
7         Brokerage Companies                                          11
8         Securities Registrars                                         3
9         Stock Exchanges                    